# 任務 2：建立您的第一個 AI 代理程式

在此筆記本中，您將了解如何使用 Strands Agents 架構來建立 AI 代理程式。您將先建立一個對話式代理程式，接著新增工具以提升其能力。最後，您會建立一個實用的食譜助理，可在網路上搜尋烹飪資訊。您將使用 Amazon Bedrock 搭配 Amazon Nova Lite 模型來驅動您的代理程式。

AI 代理程式與傳統 LLM 不同，因為它們能夠自發採取動作、使用工具，並朝目標運作。代理程式不僅能回答問題，還能實際為您執行任務。

#### 情境
您在 AnyCompany 工作，這是一家成長中的科技公司，希望探索 AI 代理程式如何協助自動化日常任務。您的團隊想了解這些代理程式如何在研究與客戶支援等任務方面進行協助。您將從建立智慧代理程式的基礎開始學習。

## 任務 2.1：環境設定

在此任務中，您將透過安裝必要的套件來設定環境，開始建立 AI 代理程式。

In [ ]:
# Install the Strands Agents framework and tools
# %pip install strands-agents strands-agents-tools

## 任務 2.2：建立您的第一個 AI 代理程式

在此任務中，您將建立一個能進行對話的 AI 代理程式。此代理程式使用 Amazon Bedrock 搭配 Amazon Nova Lite 模型，以理解並回應您的訊息。系統提示會定義代理程式的預期行為。

In [ ]:
import warnings
warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow") 

from strands import Agent

# Create your first AI agent
agent = Agent(callback_handler=None,
    model="amazon.nova-lite-v1:0",
    system_prompt="You are a helpful assistant that provides concise responses."
)

接著，您將傳送一則訊息以測試您的代理程式。

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **注意**：Strands 架構預設使用 Amazon Bedrock 搭配 Amazon Nova Lite 模型。此模型支援對話式互動，且可透過工具加強功能。

In [ ]:
# Send a message to the agent
response = agent("Hello! Tell me a joke.")
print(response)

## 任務 2.3：為您的代理程式新增工具

在此任務中，您將為代理程式新增工具，使其能執行更多功能，而不僅限於對話。

- 您將新增由 Strands Agents SDK 提供的計算器工具。

- 您將使用 @tool 裝飾器建立一個天氣工具。

<i aria-hidden="true" class="fas fa-sticky-note" style="color:#563377"></i> **注意**：天氣工具僅為預留位置範例，將一律回傳「晴朗」。

In [ ]:
from strands import Agent, tool
from strands_tools import calculator

# Create a weather tool
@tool
def weather():
    """Get current weather information"""
    return "Sunny and O degree Celsius"

# Create an agent with tools
agent_with_tools = Agent(callback_handler=None,
    model="amazon.nova-lite-v1:0",
    tools=[calculator, weather],
    system_prompt="You are a helpful assistant. You can do math calculations and use the weather tool to tell the weather."
)

# Test the agent with both tools in one query
# Note: This query requires both tools - the weather tool to get temperature in Celsius,
# and the calculator tool to convert from Celsius to Fahrenheit.
# The agent will automatically determine which tools to use and in what order.
response = agent_with_tools("What is the weather in Seattle in Fahrenheit?")
print(response)

現在來探索代理程式是如何分析問題，並決定只使用計算器工具的。

In [ ]:
# Test with a math question that only needs the calculator tool
# The agent will analyze the question and decide to use only the calculator tool
math_query = "What is 25 * 4 + 18?"
print("=== Agent Chooses Calculator Tool ===")
print(f"Query: {math_query}")
response = agent_with_tools(math_query)
print(f"Response: {response}")

### 直接工具叫用

您也可以直接呼叫工具，而不透過代理程式對話。這在測試時或當您想使用特定工具功能時非常有用。

**直接工具叫用的規格：**
- 使用 `agent.tool.tool_name()` 直接呼叫特定工具
- 將所需參數作為函數引數傳遞
- 此方式會略過代理程式的自然語言處理與工具選擇邏輯
- 適用於以程式方式存取工具功能

In [ ]:
# Call the calculator tool directly
# Note: Direct tool invocation bypasses the agent's conversation flow.
# Use agent_with_tools.tool.calculator() to call the calculator tool directly
# without the agent deciding which tool to use. This is useful for testing
# specific tools or when you know exactly which tool function you need.
result = agent_with_tools.tool.calculator(expression="2 + 3 * 4")
print(f"Calculator result: {result}")

## 任務 2.4：設定記錄

在此任務中，您將設定記錄，以了解代理程式在背景中執行的操作。這有助於您了解代理程式如何處理請求與使用工具。

Strands 架構使用 Python 的標準記錄模組，以提供代理程式運作的可見性。您可以設定不同的日誌層級，以取得更多或更少關於代理程式活動的詳細資訊。

In [ ]:
import logging
from strands import Agent
import os

# Enable detailed logging to understand what the agent is doing
logging.getLogger("strands").setLevel(logging.INFO)

# Set up logging to write to both console and file
logging.basicConfig(
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    level=logging.INFO,
    handlers=[
        logging.StreamHandler()  # Console output
    ]
)

# Create a logger
logger = logging.getLogger("agent_activity")

# Create an agent with logging enabled
logger.info("Creating new agent with Nova Lite model")
logged_agent = Agent(callback_handler=None,model="amazon.nova-lite-v1:0")

logger.info("Sending message to agent: 'Hello! How are you?'")
response = logged_agent("Hello! How are you?")
print(response)

## 任務 2.5：探索模型組態

在此任務中，您將學習如何為代理程式設定不同的模型與設定。您可以指定要使用的 AI 模型，並透過像溫度這樣的參數來調整其行為。溫度會控制回應的創造性或一致性。較低的溫度值會使回應更一致且可預測，而較高的值則會讓回應更具創造性與多樣性。

In [ ]:
from strands import Agent
from strands.models import BedrockModel

# Create a custom model configuration
custom_model = BedrockModel(
    model_id="amazon.nova-lite-v1:0",
    temperature=0.3  # Lower temperature = more consistent responses
)

# Create an agent with the custom model
custom_agent = Agent(callback_handler=None,model=custom_model)
print("Agent created successfully!")

## 任務 2.6：建立食譜助理代理程式

在此任務中，您將建立一個更實用的代理程式，可協助處理烹飪相關的工作。此食譜助理可在網路上搜尋食譜與烹飪資訊，展示代理程式在真實世界情境中的實用性。

首先，安裝食譜代理程式所需的網路搜尋套件。

In [ ]:
%pip install ddgs

建立一個網路搜尋工具，讓食譜代理程式可用來搜尋烹飪資訊與線上食譜。

In [ ]:
from strands import Agent, tool
from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException
import logging

# Set up logging
logging.getLogger("strands").setLevel(logging.INFO)

# Create a web search tool
@tool
def websearch(keywords: str, max_results: int = 3) -> str:
    """Search the web for information.
    Args:
        keywords (str): What to search for
        max_results (int): How many results to return
    Returns:
        Search results as text
    """
    try:
        results = DDGS().text(keywords, max_results=max_results)
        return results if results else "No results found."
    except Exception as e:
        return f"Search error: {e}"

print("Web search tool created successfully!")

接著，建立使用該網路搜尋工具的食譜助理代理程式，以協助回答烹飪問題。

In [ ]:

# Create the recipe assistant agent
recipe_agent = Agent(callback_handler=None,
    model="amazon.nova-lite-v1:0",
    system_prompt="""You are RecipeBot, a helpful cooking assistant.
    Help users find recipes and answer cooking questions.
    Use the websearch tool to find recipes and cooking information.""",
    tools=[websearch]
)

print("Recipe assistant agent created successfully!")

測試您的食譜助理，向它詢問烹飪相關的問題。觀察它如何使用網路搜尋工具以取得最新資訊。

In [ ]:
# Test the recipe assistant
response = recipe_agent("Suggest a simple recipe with chicken and broccoli.")
print(response)

您已成功建立一個可在網路上搜尋並協助解答烹飪問題的實用 AI 代理程式。這展示了代理程式如何在真實世界任務中發揮作用。

您現在已實際操作過 Strands Agents 架構，此架構可讓您建立能使用工具並採取動作的 AI 代理程式。透過此架構，您已了解代理程式與傳統 LLM 的不同之處，代理程式能主動使用工具來完成任務。

### 自行嘗試
- 修改系統提示，以建立適用於不同使用案例的代理程式。
- 為團隊可能需要的特定任務建立自訂工具。
- 嘗試不同的模型組態，以了解其對代理程式行為的影響。

### 清理

您已完成此筆記本。若要移至實驗室的下一部分，請執行以下操作：

- 關閉此筆記本檔案，並繼續進行**結論**。